# Model Evaluation — XGBoost Fraud Detection

**Prerequisite:** Run `python src/train.py` first to generate `models/xgb_fraud_v1.pkl`  
**What this notebook covers:**
- ROC curve + Precision-Recall curve
- Threshold sweep (find optimal operating point)
- Top 20 feature importances
- Score distribution: fraud vs legit
- MLflow run summary


In [ ]:
import pickle
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
from pathlib import Path
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    average_precision_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT = Path('..').resolve()
MODEL_PATH = ROOT / 'models' / 'xgb_fraud_v1.pkl'
DB_PATH    = ROOT / 'data' / 'fraud.duckdb'

# Load model artifact
with open(MODEL_PATH, 'rb') as f:
    artifact = pickle.load(f)

model        = artifact['model']
feature_cols = artifact['feature_cols']
print(f'Model loaded: {len(feature_cols)} features')
print(f'Model type: {type(model).__name__}')

In [ ]:
# Load fraud_features from DuckDB
con = duckdb.connect(str(DB_PATH), read_only=True)
df  = con.execute('SELECT * FROM fraud_features ORDER BY transaction_dt ASC').df()
con.close()
print(f'Loaded: {len(df):,} rows x {df.shape[1]} cols')

# Re-encode categoricals (same logic as train.py)
CATEGORICAL_FEATURES = ['product_cd', 'card4', 'card6', 'device_type']
for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].fillna('MISSING').astype(str))

# Build feature matrix
available_cols = [c for c in feature_cols if c in df.columns]
X = df[available_cols]
y = df['is_fraud']

# Time-based holdout: last 20%
n     = len(X)
split = int(n * 0.8)
X_hold, y_hold = X.iloc[split:], y.iloc[split:]
proba = model.predict_proba(X_hold)[:, 1]

print(f'\nHoldout set: {len(y_hold):,} rows')
print(f'Fraud in holdout: {y_hold.sum():,} ({y_hold.mean():.2%})')

## 1. ROC Curve & Precision-Recall Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_hold, proba)
roc_auc     = auc(fpr, tpr)
prec, rec, _ = precision_recall_curve(y_hold, proba)
ap           = average_precision_score(y_hold, proba)

print(f'Holdout AUC-ROC:       {roc_auc:.4f}')
print(f'Holdout Avg Precision: {ap:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax = axes[0]
ax.plot(fpr, tpr, color='#DD8452', lw=2, label=f'XGBoost (AUC = {roc_auc:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random')
ax.fill_between(fpr, tpr, alpha=0.1, color='#DD8452')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Holdout Set', fontweight='bold')
ax.legend()

# Precision-Recall
ax = axes[1]
ax.plot(rec, prec, color='#4C72B0', lw=2, label=f'XGBoost (AP = {ap:.4f})')
ax.axhline(y_hold.mean(), color='gray', linestyle='--', lw=1.5,
           label=f'Baseline fraud rate ({y_hold.mean():.2%})')
ax.fill_between(rec, prec, alpha=0.1, color='#4C72B0')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Holdout Set', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## 2. Threshold Sweep — Finding the Optimal Operating Point

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    preds = (proba >= t).astype(int)
    tp = ((preds==1) & (y_hold==1)).sum()
    fp = ((preds==1) & (y_hold==0)).sum()
    fn = ((preds==0) & (y_hold==1)).sum()
    p  = tp/(tp+fp) if (tp+fp)>0 else 0
    r  = tp/(tp+fn) if (tp+fn)>0 else 0
    f1 = 2*p*r/(p+r) if (p+r)>0 else 0
    precisions.append(p); recalls.append(r); f1s.append(f1)

best_idx = int(np.argmax(f1s))
best_t   = thresholds[best_idx]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(thresholds, precisions, label='Precision', color='#4C72B0', lw=2)
ax.plot(thresholds, recalls,    label='Recall',    color='#DD8452', lw=2)
ax.plot(thresholds, f1s,        label='F1 Score',  color='#2ca02c', lw=2.5)
ax.axvline(best_t, color='red', linestyle='--', lw=1.5,
           label=f'Best F1 @ threshold={best_t:.3f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Sweep: Precision / Recall / F1', fontweight='bold')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print(f'Best threshold: {best_t:.3f}')
print(f'  Precision: {precisions[best_idx]:.3f}')
print(f'  Recall:    {recalls[best_idx]:.3f}')
print(f'  F1 Score:  {f1s[best_idx]:.3f}')
print(f'\nAt threshold 0.5:')
idx50 = np.argmin(np.abs(thresholds - 0.5))
print(f'  Precision: {precisions[idx50]:.3f}  Recall: {recalls[idx50]:.3f}  F1: {f1s[idx50]:.3f}')

## 3. Top 20 Feature Importances

In [ ]:
fi = pd.Series(model.feature_importances_, index=available_cols).sort_values(ascending=False)
top20 = fi.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
median_importance = top20.median()
colors = ['#d62728' if v > median_importance else '#4C72B0' for v in top20.values]
ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Top 20 Feature Importances — XGBoost Fraud Model', fontweight='bold')
ax.axvline(median_importance, color='gray', linestyle='--', lw=1, alpha=0.7, label='Median')
ax.legend()
plt.tight_layout()
plt.show()

print('Top 10 features:')
for feat, imp in fi.head(10).items():
    print(f'  {feat:<40} {imp:.4f}')

## 4. Score Distribution: Fraud vs Legitimate

In [ ]:
fraud_scores = proba[y_hold.values == 1]
legit_scores = proba[y_hold.values == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(legit_scores, bins=100, alpha=0.6, color='#4C72B0', density=True, label=f'Legit (n={len(legit_scores):,})')
ax.hist(fraud_scores, bins=100, alpha=0.7, color='#DD8452', density=True, label=f'Fraud (n={len(fraud_scores):,})')
ax.set_xlabel('Model Score (predicted probability)')
ax.set_ylabel('Density')
ax.set_title('Score Distribution: Fraud vs Legit', fontweight='bold')
ax.legend()

ax = axes[1]
ax.hist(legit_scores, bins=100, alpha=0.6, color='#4C72B0', density=True, label='Legit')
ax.hist(fraud_scores, bins=100, alpha=0.7, color='#DD8452', density=True, label='Fraud')
ax.set_xscale('log')
ax.set_xlabel('Model Score (log scale)')
ax.set_ylabel('Density')
ax.set_title('Score Distribution (Log Scale)', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Fraud  median score: {np.median(fraud_scores):.4f}')
print(f'Legit  median score: {np.median(legit_scores):.4f}')
print(f'Separation ratio:    {np.median(fraud_scores)/np.median(legit_scores):.1f}x')
print(f'\nFraud  >0.5 capture: {(fraud_scores>0.5).mean():.1%} of fraud transactions')
print(f'Legit  >0.5 flagged: {(legit_scores>0.5).mean():.2%} false positive rate')

## 5. MLflow Run Summary

In [ ]:
import mlflow

mlflow.set_tracking_uri(str(ROOT / 'mlruns'))
client = mlflow.MlflowClient()

experiment = client.get_experiment_by_name('fraud_detection')
if experiment:
    runs = client.search_runs(
        experiment.experiment_id,
        order_by=['metrics.holdout_auc DESC']
    )
    summary = pd.DataFrame([{
        'run_id':      r.info.run_id[:10],
        'run_name':    r.info.run_name,
        'cv_auc':      r.data.metrics.get('cv_mean_auc'),
        'holdout_auc': r.data.metrics.get('holdout_auc'),
        'holdout_ap':  r.data.metrics.get('holdout_ap'),
        'n_features':  r.data.params.get('n_features'),
        'n_estimators': r.data.params.get('n_estimators'),
        'max_depth':   r.data.params.get('max_depth'),
    } for r in runs])
    print(f'Experiment: {experiment.name}  ({len(runs)} run(s))')
    print()
    print(summary.to_string(index=False))
else:
    print('No MLflow experiment found. Run python src/train.py first.')